# Quadrotor Flowpipe with Editable Inertia and PI Gains

Same flowpipe pipeline as `quadrotor_flowpipe.ipynb`, but the inner-loop
rate controller now exposes:

- **`J`** — 3×3 inertia (kg·m²), accepts scalar / length-3 / 3×3
- **`Kp`, `Ki`** — proportional and integral rate gains
- **`tau_dist`** — sup-norm bound on torque disturbance (N·m)
- **`gravity`** — was hardcoded; now editable

All math is documented in `cp_reach_notes/quadrotor_pi_rate_loop.tex`.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Add repo root to path
repo_root = None
for path in [Path.cwd().resolve()] + list(Path.cwd().resolve().parents):
    if (path / 'cp_reach').exists():
        repo_root = path
        break
if repo_root is None:
    raise RuntimeError('Could not locate repo root')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from cp_reach.development.applications.quadrotor.trajectory import find_cost_function, plan_trajectory
from cp_reach.development.applications.quadrotor.invariant import solve_nested
from cp_reach.physics import angular_acceleration as aa
from cp_reach.plotting.plotting import flowpipes, plot_flowpipes, plot_error_bounds_2d

%matplotlib inline
%load_ext autoreload
%autoreload 2
plt.rcParams['figure.dpi'] = 100

## 1. Editable parameters

All knobs in one place. Re-run downstream cells after editing.

**Inertia formats accepted by `solve_inv_set_pi`:**
- scalar `0.02` → `0.02 · I₃`
- length-3 `[Jxx, Jyy, Jzz]` → diagonal
- 3×3 ndarray → full (allows products of inertia like `J_xz`)

Same conventions for `Kp` and `Ki`.

In [ ]:
# ============================================================
# Quadrotor physical parameters
# ============================================================
# Diagonal inertia of a small ~500g racing quadrotor (kg*m^2)
J = np.diag([0.0123, 0.0123, 0.0224])

# To use a full inertia matrix instead, uncomment:
# J = np.array([
#     [0.0123, 0.0,    0.0008],
#     [0.0,    0.0123, 0.0   ],
#     [0.0008, 0.0,    0.0224],
# ])

gravity = 9.81  # m/s^2

# ============================================================
# Inner rate-loop PI gains (units: N*m per rad/s, N*m per rad)
# ============================================================
Kp = 0.5    # scalar -> 0.5*I_3; or pass array_like
Ki = 0.05   # set to 0.0 for P-only

# ============================================================
# Disturbances
# ============================================================
tau_dist   = 0.05   # N*m   -- max torque disturbance per axis
vel_dist   = 0.0    # m/s   -- velocity disturbance
accel_dist = 0.0    # m/s^2 -- linear acceleration disturbance

pid_values = {
    'J':        J,
    'Kp':       Kp,
    'Ki':       Ki,
    'tau_dist': tau_dist,
}

# Quick sanity check on the inner-loop solver in isolation
dynamics_sol = aa.solve_inv_set_pi(J=J, Kp=Kp, Ki=Ki, tau_dist=tau_dist)
print(f"Inner-loop PI LMI solved:")
print(f"  alpha       = {dynamics_sol['alpha']:.4f}")
print(f"  mu1         = {dynamics_sol['mu1']:.4e}")
print(f"  ||omega||_max approx = sqrt(mu1)*tau_dist = {np.sqrt(dynamics_sol['mu1'])*tau_dist:.4f} (Q-norm units)")

## 2. Plan polynomial trajectory

Identical to the baseline notebook — minimum-snap through 7 waypoints.

In [ ]:
num_coords = 7
n_legs = num_coords - 1
poly_deg = 7
min_deriv = 4
bc_deriv = 4

pos = [
    [0, 0, 0],
    [7.04, -0.76, 0],
    [10.04, 1.7, 0],
    [10.22, 6.6, 0],
    [13.33, 8.65, 0],
    [20.15, 8.14, 0],
    [19.6, -1.92, 0],
]
vel = [
    [0, 0, 0],
    [2.37, 0, 0],
    [0.15, 2.67, 0],
    [0.49, 2.28, 0],
    [2.85, -0.23, 0],
    [0, 0, 0],
    [0, 0, 0],
]
acc  = [[0, 0, 0] for _ in range(num_coords)]
jerk = [[0, 0, 0] for _ in range(num_coords)]

bc = np.stack((pos, vel, acc, jerk))
k_time = 1e5
T_legs = [4.67, 2.17, 1.84, 1.92, 5.5, 6.46]

cost = find_cost_function(
    poly_deg=poly_deg, min_deriv=min_deriv,
    rows_free=[], n_legs=n_legs, bc_deriv=bc_deriv,
)
ref = plan_trajectory(bc, cost, n_legs, poly_deg, k_time, T_legs)
print(f"Trajectory planned with {len(ref['x'])} points")

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.plot(ref['x'], ref['y'], ref['z'], 'b-', linewidth=2, label='Trajectory')
ax.scatter(*zip(*pos), c='r', s=100, marker='o', label='Waypoints')
ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)'); ax.set_zlabel('Z (m)')
ax.set_title('Quadrotor Trajectory')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 3. Compute invariant set with PI rate loop

`solve_nested` now uses the new `pid_values` path — inner layer is the
6-state augmented PI system in `(ω, ∫ω)`, projected back to ω via Schur
complement before being fed to the SE(2,3) layer.

In [ ]:
print('Solving for invariant set with PI rate loop...')
print(f"  J diag:     {np.diag(np.atleast_2d(J)) if np.ndim(J) > 0 else J}")
print(f"  Kp:         {Kp}")
print(f"  Ki:         {Ki}")
print(f"  tau_dist:   {tau_dist} N*m")
print(f"  vel_dist:   {vel_dist} m/s")
print(f"  accel_dist: {accel_dist} m/s^2")
print(f"  gravity:    {gravity} m/s^2")

result = solve_nested(
    vel_dist=vel_dist,
    accel_dist=accel_dist,
    ref=ref,
    pid_values=pid_values,
    gravity=gravity,
)

angular_result = result['angular']
se3_result     = result['se23']

ang_vel_points = angular_result['points']
angular_bounds = angular_result['bounds']
omega_dist     = angular_result['omega_dist']
dynamics_sol   = angular_result['lmi_solution']

xi_points       = se3_result['points_algebra']
eta_points      = se3_result['points_group']
xi_bounds       = se3_result['bounds_algebra']
eta_bounds      = se3_result['bounds_group']
kinematics_sol  = se3_result['lmi_solution']

print('\nInvariant set computed:')
print(f"  omega bound (rad/s):           {angular_bounds.flatten()}")
print(f"  omega_dist passed to outer:    {omega_dist:.4f}")
print(f"  Position bounds (Lie algebra): {xi_bounds[:3, 0]} to {xi_bounds[:3, 1]}")
print(f"  Position bounds (Lie group):   {eta_bounds[:3, 0]} to {eta_bounds[:3, 1]}")

## 4. Visualize invariant set

In [ ]:
fig = plot_error_bounds_2d(xi_points, eta_points)

## 5. Flowpipes along the trajectory

In [ ]:
print('Computing flowpipes...')
flowpipes_list, nominal_traj = flowpipes(
    ref=ref,
    step=2,
    vel_dist=vel_dist,
    accel_dist=accel_dist,
    omega_dist=omega_dist,
    sol=kinematics_sol,
    axis='xy',
)
print(f'Generated {len(flowpipes_list)} flowpipe segments')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
plot_flowpipes(nominal_traj, flowpipes_list, ax=ax, axis='xy')
ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
ax.set_title(f'Quadrotor Flowpipe — PI rate loop (Kp={Kp}, Ki={Ki}, tau_dist={tau_dist} N*m)')
ax.grid(True, alpha=0.3)
ax.axis('equal')
plt.tight_layout(); plt.show()

## 6. Quick sensitivity sweep over `Kp`

Holds everything else fixed and sweeps the proportional rate gain. Larger
`Kp` should tighten the ω-bound (faster rejection) but eventually saturate
as the LMI `mu1` floor is set by the disturbance entry channel `J⁻¹`.

In [ ]:
Kp_sweep = np.linspace(0.1, 2.0, 12)
mu1_vals = []
omega_norm_vals = []

for kp in Kp_sweep:
    sol = aa.solve_inv_set_pi(J=J, Kp=kp, Ki=Ki, tau_dist=tau_dist)
    mu1_vals.append(sol['mu1'])
    omega_norm_vals.append(np.sqrt(sol['mu1']) * tau_dist)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(Kp_sweep, mu1_vals, 'o-')
axes[0].set_xlabel('Kp'); axes[0].set_ylabel('mu1')
axes[0].set_yscale('log'); axes[0].grid(True, alpha=0.3)
axes[0].set_title('LMI gain mu1 vs Kp')

axes[1].plot(Kp_sweep, omega_norm_vals, 'o-', color='C1')
axes[1].set_xlabel('Kp'); axes[1].set_ylabel(r'$\sqrt{\mu_1}\,\tau_{max}$  (Q-norm of $\omega$)')
axes[1].grid(True, alpha=0.3)
axes[1].set_title('omega bound vs Kp')

plt.tight_layout(); plt.show()

## 7. Summary

Differences from `quadrotor_flowpipe.ipynb`:

| Knob | Original | This notebook |
|------|----------|---------------|
| Inertia `J` | implicit `I₃` | parameter |
| Rate gains | `Kdq=10` (P-only, hardcoded) | `Kp, Ki` parameters |
| Disturbance unit | `ang_accel_dist` (rad/s²) | `tau_dist` (N·m) |
| Gravity | hardcoded `9.8` | `gravity` kwarg |
| Solver path | `solve_inv_set(Kdq)` | `solve_inv_set_pi(J, Kp, Ki, tau_dist)` |
